In [1]:
# !pip install accelerate rouge kiwipiepy lm-eval

In [2]:
# 정량적 평가 BLEU(기계번역 성능평가)  ROUGE
# BLEU 예측문장과 정답문장이 얼마나 많은 n-gram을 공유
# BLUE-1 : unigram
# 단점 : 동의어를 인식 못함  The car is fast  /  The automobile is quick

# ROUGE(문서요약 평가)
# n-gram을 비교.. Recall 중심  The cat sat on the mat / The cat sat
# 정답 6, 예측 3   6/3 = 0.5 -- recall    3 / 3 ->1.0  Precision
# ROUGE-1 : unigram..

# 최근에는 다양한 평가지표가 사용 BERTScore...LLM-as-a-juge

In [6]:
from nltk.translate.bleu_score import sentence_bleu,SmoothingFunction
from rouge import Rouge
from kiwipiepy import Kiwi

Kiwi = Kiwi()  # 한국어 토큰화
# 한국어 형태소 단위 분리 함수
def tokenize_korean(text):
    return [token.form for token in Kiwi.tokenize(text)]

# 스무딩기법 점수가 불필요하게 0이되는 문제를 완화하기위해 사용
def evaluate_ngram_improved(reference, candidate):
    smothie =  SmoothingFunction().method4
    ref_tokens = [tokenize_korean(reference)]
    cand_tokens = tokenize_korean(candidate)

    blue_score = sentence_bleu(ref_tokens,cand_tokens,smoothing_function=smothie)
    rouge = Rouge()
    ref_str = ' '.join(ref_tokens[0])
    cand_str = ' '.join(cand_tokens)
    scores = rouge.get_scores(cand_str,ref_str)[0]
    return blue_score, scores

ref = '파리를 여항할 때는 에펠탑과 루부르 박물관을 꼭 방문해야 합니다.'
cand = '파리 여행 시 에펠탑과 루브르 박물관은 반드시 가봐야 할 명소입니다.'

bleu, rouge_rs = evaluate_ngram_improved(ref,cand)

print(f'BLUE : {bleu}')
print(f"ROUGE-1 F1: {rouge_rs['rouge-1']['f']:.4f}")

BLUE : 0.06399862641607625
ROUGE-1 F1: 0.4706


In [5]:
rouge_rs

{'rouge-1': {'r': 0.5, 'p': 0.4444444444444444, 'f': 0.4705882303114187},
 'rouge-2': {'r': 0.17647058823529413,
  'p': 0.17647058823529413,
  'f': 0.17647058323529427},
 'rouge-l': {'r': 0.4375, 'p': 0.3888888888888889, 'f': 0.41176470089965406}}

In [10]:
# LM-Evaluation_Harness벤치 마크 평가
# 모델의 추론 능력을 벤치마트 데이터셋(hellaswag)을 통해 accuracy를 측정
import lm_eval
try:
    result = lm_eval.simple_evaluate(
        model="hf",
        model_args = "pretrained=skt/kogpt2-base-v2,dtype=float32",
        tasks=['hellaswag'],
        device="cpu",
        limit=2
    )
    print(f"LM-Eval 정확도(acc): {result['results']['hellaswag']['acc,none']}")
except Exception as e:
    print(e)

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Map:   0%|          | 0/39905 [00:00<?, ? examples/s]

Map:   0%|          | 0/10042 [00:00<?, ? examples/s]

Running loglikelihood requests: 100%|██████████| 8/8 [00:00<00:00,  8.86it/s]


LM-Eval 정확도(acc): 0.0


In [12]:
# 정답형태가 고정되어 있지않은 생성형 태스크의 경우 상용모델(gpt)을 판단기준으로 활용
import os
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv(override=True)
client = OpenAI()
def evaluate_with_gpt(prompt, generated_text):
    eval_prompt = f'''
다음 질문과 답변을 보고 정확성, 유창성, 관련성을 기준으로 1~5점 사이의 점수와 이류를 작성해주세요
질문:{prompt}
답변:{generated_text}
'''
    try:
        response = client.chat.completions.create(
            model = 'gpt-5.4-nano',
            messages=[{'role':'user','content':eval_prompt}],
            max_tokens = 150
        )
        return response.choice[0].message.content
    except Exception as e:
        return e
evaluate_with_gpt('프랑시의 명소는?','파리 여행시 에팔탑과 루브르 박물관은 반드시 가봐야 할 명소입니다.')    
    

openai.BadRequestError('Error code: 400 - {\'error\': {\'message\': "Unsupported parameter: \'max_tokens\' is not supported with this model. Use \'max_completion_tokens\' instead.", \'type\': \'invalid_request_error\', \'param\': \'max_tokens\', \'code\': \'unsupported_parameter\'}}')